# 00 - Setup and Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AfefGuesmi/msdd-arab-youth-space-hackathon/blob/main/MSDD-Net/notebooks/00_Setup_and_Data.ipynb)

Installs dependencies, fetches the repository, downloads the prepared datasets, and verifies the folder layout.

No GPU required.

## 1. Environment check

In [ ]:
import sys, platform
print('Python', sys.version)
print('Platform', platform.platform())

try:
    import torch
    print('PyTorch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except ImportError:
    print('PyTorch not installed yet.')

## 2. Clone repository

In [ ]:
import os
REPO_URL = 'https://github.com/AfefGuesmi/msdd-arab-youth-space-hackathon.git'
BRANCH   = 'msddnet'

if not os.path.isdir('msdd-arab-youth-space-hackathon'):
    !git clone -b {BRANCH} {REPO_URL}

%cd msdd-arab-youth-space-hackathon/MSDD-Net
!git pull
!ls

## 3. Install dependencies

In [ ]:
# CUDA build of PyTorch (Colab already has this, so it is a no-op there)
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121

# Rest of the dependencies
!pip install -q -r requirements.txt

print('Dependencies installed.')

## 4. Download data

Two options:

- **Google Drive** (fast, recommended): set the folder IDs below. The notebook uses `gdown` to fetch the zip files.
- **Zenodo** (public): the notebook falls back to the Zenodo URLs if the Drive IDs are empty.

In [ ]:
# Fill these in with your own Google Drive folder IDs.
# Leave empty to fall back to Zenodo.
DRIVE_MARIDA_FOLDER = ''  # e.g. '1AbCdEfGhIjKlMnOpQrStUvWxYz'
DRIVE_MADOS_FOLDER  = ''  # optional, only needed for MADOS training

import os, subprocess

def shell(cmd):
    print('$', cmd)
    return subprocess.call(cmd, shell=True)

!pip install -q gdown

In [ ]:
# ---- MARIDA ----
os.makedirs('data', exist_ok=True)

if DRIVE_MARIDA_FOLDER:
    shell(f"gdown --folder {DRIVE_MARIDA_FOLDER} -O data/_marida_dl")
    # Move whatever the folder contained into data/ directly if needed
    for root, dirs, files in os.walk('data/_marida_dl'):
        for d in dirs:
            if d in ('patches', 'splits'):
                src = os.path.join(root, d)
                dst = os.path.join('data', d)
                if not os.path.exists(dst):
                    shell(f'mv "{src}" "{dst}"')
else:
    print('No Drive ID set for MARIDA. Download manually from https://zenodo.org/record/5151941 and unpack into ./data/')

In [ ]:
# ---- MADOS (optional) ----
if DRIVE_MADOS_FOLDER:
    shell(f"gdown --folder {DRIVE_MADOS_FOLDER} -O ../MADOS_nearest")
else:
    print('No Drive ID set for MADOS. Only needed for train_on_mados.py / evaluate_on_mados.py.')

## 5. Verify the folder layout

In [ ]:
import os

checks = [
    ('data/patches', os.path.isdir('data/patches')),
    ('data/splits',  os.path.isdir('data/splits')),
    ('data/splits/train_X.txt', os.path.isfile('data/splits/train_X.txt')),
    ('data/splits/val_X.txt',   os.path.isfile('data/splits/val_X.txt')),
    ('data/splits/test_X.txt',  os.path.isfile('data/splits/test_X.txt')),
]

ok = True
for name, present in checks:
    print(f"{'OK ' if present else 'MISSING'} {name}")
    ok = ok and present

if ok:
    import numpy as np
    for split in ('train', 'val', 'test'):
        rois = np.genfromtxt(f'data/splits/{split}_X.txt', dtype='str', ndmin=1)
        print(f"{split}: {len(rois)} patches")
else:
    print('\nSome files are missing. Re-run the download cells above.')

## 6. Class distribution (optional sanity check)

In [ ]:
# MARIDA's published 15-class distribution from dataloader.py
import matplotlib.pyplot as plt
import numpy as np

CLASS_DISTR_15 = [0.00452, 0.00203, 0.00254, 0.00168, 0.00766,
                  0.15206, 0.20232, 0.35941, 0.00109, 0.20218,
                  0.03226, 0.00693, 0.01322, 0.01158, 0.00052]
ids = list(range(1, 16))

fig, ax = plt.subplots(figsize=(11, 7))
colors = ['#f4513a' if i == 6 else '#1f6e8c' for i in ids]
ax.barh(ids, [v * 100 for v in CLASS_DISTR_15], color=colors, height=0.7)
ax.set_yticks(ids)
ax.invert_yaxis()
ax.set_xscale('log')
ax.set_xlim(0.01, 100)
ax.set_xticks([0.01, 0.1, 1, 10, 100])
ax.set_xticklabels(['0.01%', '0.1%', '1%', '10%', '100%'])
ax.set_title('Share of annotated pixels per class (log scale)')
ax.grid(axis='x', color='#dddddd')
ax.set_axisbelow(True)
for s in ('top','right','left'):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

## Next steps

- Run **01_Random_Forest_Baseline.ipynb** for the fast CPU baseline.
- Run **02_UNet_Baseline.ipynb** for the MARIDA paper U-Net.
- Run **03_MSDDNet_Training.ipynb** for the transformer model.
- Run **04_Evaluation_and_Figures.ipynb** to reproduce the paper tables and figures.